# Data Cleaning — Craigslist Used Cars Dataset
Based on findings from Exploratory Data Analysis (`eda.ipynb`).

In [1]:
import pandas as pd
import numpy as np
import os

DATA_PATH = 'dataset\\vehicles.csv'
CLEAN_DATA_PATH = 'dataset\\vehicles_cleaned.csv'

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Original shape: {df.shape}')

Original shape: (426880, 26)


## 1. Drop Unnecessary Columns
Columns like `county` (100% missing), and identifiers/URLs (`id`, `url`, `region_url`, `VIN`, `image_url`) are dropped as they do not provide useful predictive power for modeling.

In [2]:
cols_to_drop = ['county', 'id', 'url', 'region_url', 'VIN', 'image_url']
df = df.drop(columns=cols_to_drop, errors='ignore')
print(f'Shape after dropping columns: {df.shape}')

Shape after dropping columns: (426880, 20)


## 2. Handle Outliers (Based on IQR Method from EDA)
Based on the analysis in the EDA phase:
- **`price`**: Fence is `[-24,978.6, 57,364.4]`. We will remove $0 cars and those above the upper fence.
- **`year`**: Fence is `[1,994.5, 2,030.5]`. We will keep cars with a year >= 1995.
- **`odometer`**: Fence is `[-106,053.8, 277,300.2]`. We will keep cars with an odometer reading <= 277,300.

In [3]:
# Filter Price (Removing $0 cars and outliers above the fence)
df = df[(df['price'] > 0) & (df['price'] <= 57364.4)]

# Filter Year (Keeping year >= 1995)
df = df[df['year'] >= 1995]

# Filter Odometer (Keeping odometer <= 277,300)
df = df[df['odometer'] <= 277300]

print(f'Shape after removing outliers: {df.shape}')

Shape after removing outliers: (364070, 20)


## 3. Handle Missing Values
For critical categorical and numerical columns that have a small percentage of missing values, we can drop the missing rows. For other categorical columns with many missing values, we'll impute them with `'unknown'`.

In [4]:
# Drop rows missing critical features
critical_cols = ['manufacturer', 'model', 'fuel', 'transmission', 'title_status']
df = df.dropna(subset=critical_cols)

# Impute remaining categorical missing values with 'unknown'
categorical_cols = df.select_dtypes(include=['object']).columns
df[categorical_cols] = df[categorical_cols].fillna('unknown')

# For continuous features that may still have missing values, fill with median
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

print(f'Shape after handling missing values: {df.shape}')
print(f'Remaining missing values:\n{df.isnull().sum()}')

Shape after handling missing values: (340224, 20)
Remaining missing values:
region          0
price           0
year            0
manufacturer    0
model           0
condition       0
cylinders       0
fuel            0
odometer        0
title_status    0
transmission    0
drive           0
size            0
type            0
paint_color     0
description     0
state           0
lat             0
long            0
posting_date    0
dtype: int64


## 4. Save Cleaned Dataset
Finally, we save the cleaned dataframe to a new CSV file.

In [5]:
df.to_csv(CLEAN_DATA_PATH, index=False)
print(f'Cleaned dataset successfully saved to {CLEAN_DATA_PATH}')

Cleaned dataset successfully saved to dataset\vehicles_cleaned.csv
